In [0]:


with aa (
    select country, year, brand_name, model, d_type, category,
        
        count(sc_measurement) total_count,
        avg(sc_measurement) avg_sc,
        count(case when sc_measurement = 1 then sc_measurement else null end) positive,
        count(case when sc_measurement = 0 then sc_measurement else null end) neutral,
        count(case when sc_measurement = -1 then sc_measurement else null end) negative
    
    from (
          SELECT
        `country`,
        `year`,
        case when `brand_name`='SS' then 'Samsung' else `brand_name` end as brand_name, 
        `model`,
        case when device_type = 'OLED' then 'OLED'
        when device_type = 'UHD' then 'UHD'
        else 'QLED/QNED/Nanocell'
        end as d_type, 
        cate_2_depth as category, 
        sc_measurement, 
        memo
        -- count(sc_measurement) total_count,
        -- avg(sc_measurement) avg_sc,
        -- count(case when sc_measurement = 1 then sc_measurement else null end) positive,
        -- count(case when sc_measurement = 0 then sc_measurement else null end) neutral,
        -- count(case when sc_measurement = -1 then sc_measurement else null end) negative
    FROM
        `kic_data_ods`.`buzzmetrix`.`buzzmetrix`
    WHERE 1=1
        AND `cate_2_depth` in ('07-04. SW/OS', '07-20. 전반적 스마트 사용성') 
        AND `country` in ('United States', 'United Kingdom', 'Korea', 'Germany', 'Brazil')
        AND `year` >= '2022'
        AND `brand_name` in ('Samsung', 'LG', 'SS', 'TCL')
        AND device_type not like '%FHD%'
        AND `cate_2_depth` IS NOT NULL
    )

    group by country, year, brand_name, model, d_type, category
    order by country, year, brand_name, model, d_type, category

    )

    select a.country, a.year, a.brand_name, a.model, a.d_type, a.avg_sc as avg_sw_os, b.avg_sc as avg_smart_usage, a.total_count as total_sw_os, b.total_count as total_smart_usage
    from aa a
      join  (select * from aa where category='07-20. 전반적 스마트 사용성') b on a.country=b.country and a.year=b.year and a.brand_name=b.brand_name and a.model=b.model and a.d_type=b.d_type 
    where 1=1 
    AND a.category='07-04. SW/OS'
    AND ((a.avg_sc > 0 AND b.avg_sc < 0) OR (a.avg_sc < 0 AND b.avg_sc > 0))
    AND b.total_count >= 100
    order by  a.brand_name, a.country, a.model, a.year, a.d_type



-- =============== 검증용 쿼리 
      SELECT
        `country`,
        `year`,
        case when `brand_name`='SS' then 'Samsung' else `brand_name` end as brand_name, 
        `model`,
        case when device_type = 'OLED' then 'OLED'
        when device_type = 'UHD' then 'UHD'
        else 'QLED/QNED/Nanocell'
        end as d_type, 
        cate_2_depth as category, 
        sc_measurement, 
        memo
        -- count(sc_measurement) total_count,
        -- avg(sc_measurement) avg_sc,
        -- count(case when sc_measurement = 1 then sc_measurement else null end) positive,
        -- count(case when sc_measurement = 0 then sc_measurement else null end) neutral,
        -- count(case when sc_measurement = -1 then sc_measurement else null end) negative
    FROM
        `kic_data_ods`.`buzzmetrix`.`buzzmetrix`
    WHERE 1=1
        AND `cate_2_depth` = '07-20. 전반적 스마트 사용성'
        AND `country` = 'United States'
        AND `year` = '2023'
        AND `model` = '(SS) S90C'
        AND device_type not like '%FHD%'
        AND `cate_2_depth` IS NOT NULL